# Per-tracer grains in SERPENS

Configure a grain population, inject dust from an **ejecting planet with no moon** (55-Cnc-e-like by default), reload its archive, and inspect physical budgets and size-resolved fields. Set `SYSTEM_KIND = 'moon'` below to use the original moon example instead. This small **neutral Python-backend** example includes gravity, stellar radiation pressure, and Poynting–Robertson drag. No C build is needed.

Use a Python kernel with the project's [normal requirements](../README.md#installation) plus `ipykernel` installed. Run all cells from top to bottom in a fresh kernel, with its working directory at the repository root or under `notebooks/`. Each simulation creates a new directory under `output/grain-notebook/`; existing `simdata/` is never overwritten.

All plots use Plotly. The combined MIME/HTML renderer supports notebook frontends without a Matplotlib magic. If your frontend does not display interactive output, open the saved planar/LOS HTML files in a browser, or set `pio.renderers.default = 'browser'` for a local kernel. HTML exports embed Plotly for offline use.

These are uncalibrated parameter choices and a short workflow demonstration, **not a mature torus or a survival prediction**. See the [grain guide](../docs/grains.md) for the complete API and physical limitations.

In [ ]:
from pathlib import Path
import copy
import os
import sys
import tempfile
from contextlib import contextmanager
from dataclasses import replace

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/grains.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the kernel in the SERPENS repository or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from src.visualizing.visualize import PlotlyVisualize as PlotlyVisualizer
from IPython.display import display, Markdown
from src.grains import GrainPopulation
from src.parameters import GLOBAL_PARAMETERS
from src.serpens_simulation import SerpensSimulation
from src.serpens_analyzer import SerpensAnalyzer

pio.renderers.default = 'plotly_mimetype+notebook'
pio.templates.default = 'plotly_white'
print(f'Repository: {ROOT}')

## 1. Choose populations (all dimensional inputs are SI)

The first population samples the **number** distribution `dN/da ∝ a^(-3.5)` between 0.3 and 3 micrometres. The second is monodisperse at 1 micrometre. Each independently supplies 1 kg/s: registering both means a total of 2 kg/s. Equal radius bounds select the monodisperse case; `size_slope=1` selects the logarithmic sampling case.

Launch sites cover the emitting body's surface, with outward hemisphere directions. The phenomenological speed law is `v(a) = v_ref * (a/a_ref)^(-gamma)`, clipped to the chosen bounds. `q_pr` is spectrum-averaged radiation-pressure efficiency, **not** extinction efficiency or a thermal model. All populations must select the same radiation source.

The planet preset uses the approximate masses, radii, and orbit from `resources/objects.json`'s `55-Cnc-e` entry; luminosity (0.58 solar) and launch speeds are explicit illustrative assumptions. Its 30 km/s reference launch speed is chosen near/above the planet's roughly 23 km/s isolated escape speed to make an extended cloud visible, **not** as a validated dust-launch mechanism. The moon preset retains 4 km/s. At 55-Cnc-e irradiation, sublimation may invalidate fixed-radius grains: neither these plots nor budget closure establish dust survival.

In [ ]:
# Using 55 Cnc e:
SOURCE_NAME = 'planet'
PRIMARY_NAME = 'star'
STAR_MASS_KG = 1.8e30
STAR_RADIUS_M = 6.56e8
STAR_LUMINOSITY_W = 0.58 * 3.828e26
PLANET_MASS_KG = 4.772e25
PLANET_RADIUS_M = 1.196e7
PLANET_ORBIT_M = 2.31e9
LAUNCH_SPEED_M_S = 30000.0
PLOT_LIM = 5
STEPS = 20
INTERVAL_S = 86400
SEED = 1847

distributed = GrainPopulation(
    name='synthetic size distribution',
    density_kg_m3=2000.0,
    radius_min_m=0.3e-6, radius_max_m=3e-6,
    size_slope=3.5,
    tracers_per_injection=64,
    dust_mass_per_sec=1.0,
    speed_ref_m_s=LAUNCH_SPEED_M_S,
    radius_ref_m=1e-6,
    speed_exponent=0.5, speed_min_m_s=0.5 * LAUNCH_SPEED_M_S, speed_max_m_s=2 * LAUNCH_SPEED_M_S,
    launch_altitude_m=1e4,
    radiation_source='star', luminosity_w=STAR_LUMINOSITY_W, q_pr=1.0,
    potential_v=0.0,
)
monodisperse = replace(distributed, name='synthetic 1 micrometre',
                       radius_min_m=1e-6, radius_max_m=1e-6)
populations = [distributed]
display(distributed.to_dict())

## 2. Inspect sampling and radiation response

An explicit seeded generator makes this preview repeatable; it does not consume the simulation's generator. For constant density and `q_pr`, grain mass scales as `a³` and `beta` as `a⁻¹`. `beta` compares radiation with the **selected star's** gravity, not the planet's gravity. The circumstellar `beta=0.5` blowout rule is not a general moon-ejecta escape criterion.

In [ ]:
sample = distributed.sample_radii(np.random.default_rng(SEED), count=10000)
radius_grid = np.geomspace(distributed.radius_min_m, distributed.radius_max_m, 100)
properties = distributed.properties(radius_grid, STAR_MASS_KG)
counts, radius_bins = np.histogram(sample * 1e6, bins=np.geomspace(0.3, 3.0, 25))
fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Bar(x=(radius_bins[:-1] + radius_bins[1:]) / 2, y=counts,
                     width=np.diff(radius_bins), name='Sampled sizes'), row=1, col=1)
fig.add_trace(go.Scatter(x=radius_grid * 1e6, y=properties['beta'],
                         mode='lines', name='Radiation response'), row=1, col=2)
fig.update_xaxes(type='log', title_text='Grain radius [micrometres]')
fig.update_yaxes(title_text='Sampled tracer count', row=1, col=1)
fig.update_yaxes(type='log', title_text='Radiation / stellar gravity (beta)', row=1, col=2)
fig.update_layout(height=450, showlegend=False)
fig.show()

## 3. Register the emitting body and integrate

The context manager restores the working directory and global parameters even if a run fails. Keep simulation and archive operations inside this context; do not run other simulations concurrently in the same kernel. The run function creates a fresh output directory on every call.

`object_to_source(SOURCE_NAME, grains=...)` registers the planet or moon independently of gas species. The planet preset adds only a star and planet; no dummy moon is needed. `orbit_object=SOURCE_NAME` also makes the numerical boundary relative to the correct primary: the star for planet ejecta, the planet for moon ejecta. Tracers have REBOUND `m=0`, `r=0`; physical grain masses and radii are archived separately. For each injection, `W = dust_mass_per_sec * interval_s / sum(grain_mass_kg)`. These birth weights never increase to compensate for lost grains.

We disable Lorentz forces and artificial circular-orbit fixing explicitly. REBOUNDx includes radiation pressure and PR drag together: do not add PR drag again. The 60-second integration chunk limit is illustrative, not a convergence result.

In [ ]:
@contextmanager
def archive_directory(path):
    original = Path.cwd()
    parameters = copy.deepcopy(GLOBAL_PARAMETERS.params)
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(original)
        GLOBAL_PARAMETERS.params = parameters


def run_grains(configurations, *, steps=STEPS, interval_s=INTERVAL_S, seed=SEED):
    output_root = ROOT / 'output/grain-notebook'
    output_root.mkdir(parents=True, exist_ok=True)
    path = Path(tempfile.mkdtemp(prefix='run-', dir=output_root))
    with archive_directory(path):
        GLOBAL_PARAMETERS.update(dict(
            celest={}, species=[], all_species=[], grains={}, all_grains={},
            lorentz_enabled=False, fix_source_circular_orbit=False,
            grain_seed=seed, integration_threads=2, integration_max_dt=60.0,
            gen_max=None, r_max=5.0,
        ))
        simulation = SerpensSimulation()
        simulation.add(
            m=STAR_MASS_KG, r=STAR_RADIUS_M, hash='star'
        )
        simulation.add(
            m=PLANET_MASS_KG, r=PLANET_RADIUS_M, a=PLANET_ORBIT_M, primary='star', hash='planet'
        )
        simulation.move_to_com()
        simulation.object_to_source(SOURCE_NAME, grains=configurations)
        simulation.save_to_file('simdata/archive.bin', delete_file=True)
        for _ in range(steps):
            simulation.advance_single(time=interval_s, orbit_object=SOURCE_NAME)
            simulation.serpens_iter += 1
    return path

run_path = run_grains(populations)
print(f'Archive and metadata: {run_path / "simdata"}')

## 4. Reload and check physical budgets

`timestep=-1` means the final archived snapshot, **not** minus one second. Read population IDs from the archive rather than assuming they are chemical IDs. The assertions check expected injection and closure for number, mass, and geometric area. No losses in such a short run would be unsurprising.

Impacts absorb grains without changing active bodies. `domain_exit` denotes leaving the numerical domain, not proven gravitational escape. Loss times bound a detection interval, not the exact impact time. Keep the entire `simdata` directory, including HDF5 metadata and parameter files, with the binary archive. Only load trusted archives (parameter files use pickle).

In [ ]:
with archive_directory(run_path):
    analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    population_configs = analyzer.grain_populations
    budgets = {pid: analyzer.grain_budget(-1, population_id=pid) for pid in population_configs}

rows = ['| Population | Injected [kg] | Retained [kg] | Removed [kg] |',
        '| --- | ---: | ---: | ---: |']
for pid, config in population_configs.items():
    budget = budgets[pid]
    expected_mass = config['dust_mass_per_sec'] * STEPS * INTERVAL_S
    np.testing.assert_allclose(budget['injected']['mass'], expected_mass, rtol=1e-12)
    assert len(budget['unaccounted_hashes']) == 0
    for quantity in ('number', 'mass', 'area'):
        np.testing.assert_allclose(
            budget['retained'][quantity] + budget['removed'][quantity],
            budget['injected'][quantity], rtol=1e-12, atol=0.0)
    rows.append(f"| {config['name']} | {budget['injected']['mass']:.3g} | "
                f"{budget['retained']['mass']:.3g} | {budget['removed']['mass']:.3g} |")
display(Markdown('\n'.join(rows)))
display({pid: budget['removed_by_cause'] for pid, budget in budgets.items()})

## 5. Primary-centered radial profiles

These profiles bin physical weights by cylindrical xy distance from `PRIMARY_NAME`: the **star for planet ejecta**, or the planet for moon ejecta. Bins follow the emitting body's orbit, not its physical radius. Column mass density is annulus mass divided by annulus area, in kg/m². The same call accepts `quantity='number'` or `'area'`. Budgets cover the full archive; profiles only cover their bins and any analysis selections.

In [ ]:
with archive_directory(run_path):
    analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    analyzer.load_timestep_data(-1)
    center = np.array(analyzer.sim.particles[PRIMARY_NAME].xyz)
    edges = np.linspace(0.85 * PLANET_ORBIT_M, 1.15 * PLANET_ORBIT_M, 25)
    profiles = {pid: analyzer.grain_radial_profile(
        -1, edges, quantity='mass', center=center, population_id=pid)
        for pid in population_configs}

fig = go.Figure()
for pid, profile in profiles.items():
    fig.add_trace(go.Scatter(x=profile['radii'] / STAR_RADIUS_M, y=profile['column_density'],
                             mode='lines', line_shape='hvh', name=population_configs[pid]['name']))
    np.savetxt(run_path / f'radial_mass_population_{pid}.csv',
               np.column_stack((profile['radii'], profile['integrated_weights'], profile['column_density'])),
               delimiter=',', header='radius_m,annulus_mass_kg,column_mass_kg_m-2')
fig.update_layout(
    xaxis_title=f'Cylindrical distance / {PRIMARY_NAME} radius',
    yaxis_title='Column mass density [kg/m²]',
    #title='Early ejecta distribution — not a developed torus'
)
#fig.write_html(run_path / 'radial_profiles.html', include_plotlyjs=True)
fig.show()

## 6. A size-selected density field

`radius_range` selects **physical grain sizes**, not spatial radii. This example selects submicrometre grains from the distributed population and the emitting body's source hash. DTFE uses birth weights: `W` for number, `W*m` for mass, or `W*pi*a²` for geometric area. `d=2` estimates a projected xy field; `d=3` estimates a volume field.

Projected area density is dimensionless but **not optical depth**. The scatter colors below are DTFE vertex estimates, not an extinction image. Empty selections have no estimator; insufficient or degenerate nonempty clouds raise `ValueError`. Do not invent densities or jitter coordinates to hide that condition.

In [ ]:
distributed_id = next(pid for pid, config in population_configs.items()
                      if config['name'] == distributed.name)
with archive_directory(run_path):
    analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    analyzer.load_timestep_data(-1)
    source_hash = analyzer.sim.particles[SOURCE_NAME].hash.value
    try:
        field = analyzer.grain_field(-1, quantity='area', d=2,
                                     population_id=distributed_id, source_hash=source_hash,
                                     radius_range=(0.3e-6, 1e-6))
    except ValueError as error:
        field = None
        print(f'Cannot estimate this selection: {error}')

if field is not None and field['estimator'] is not None:
    xy = (field['positions'] - np.asarray(analyzer.sim.particles[SOURCE_NAME].xyz)[:2]) / 1e6
    fig = go.Figure(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode='markers',
        marker=dict(size=5, color=field['density'], colorscale='Viridis', showscale=True,
                    colorbar=dict(title=f"Geometric area density [{field['units']}]<br>not optical depth"))))
    fig.update_layout(height=650, xaxis_title=f'x relative to {SOURCE_NAME} [10⁶ m]',
                      yaxis_title=f'y relative to {SOURCE_NAME} [10⁶ m]')
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.show()
elif field is not None:
    print('No surviving grains in the requested selection.')

### 6a. Visualizer: planar view

The project's `PlotlyVisualize` is imported as `PlotlyVisualizer` here. We use `add_grain_field`, not the gas density plotting path, to retain physical grain weights and SI labels. Both views below use the same **3D mass-density** estimates for the distributed population, with identical logarithmic color limits. Unlike the preceding `d=2` example, these colors represent local volume density, not projected column density.

`reference_system=SOURCE_NAME` selects the emitting body's **primary** as plot center: `planet` means star-centered, while `moon` means planet-centered. `lim` is in that primary's radii (five stellar radii or eight planet radii here). The planar view shows xy coordinates with body outlines; this backend does not add the Matplotlib source-orbit overlay. Hover, pan, and zoom using Plotly's controls. Plotting stays inside the archive context so celestial-body metadata can be read. Both views are saved as standalone interactive HTML files (no Kaleido needed). After changing `SYSTEM_KIND`, restart the kernel and run all cells so archives and plotting references stay consistent.

In [ ]:
with archive_directory(run_path):
    visualizer_analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    try:
        volume_field = visualizer_analyzer.grain_field(
            -1, quantity='mass', d=3, population_id=distributed_id, source_hash=source_hash)
    except ValueError as error:
        volume_field = None
        print(f'Cannot estimate this selection: {error}')

    if volume_field is not None and volume_field['estimator'] is not None:
        positive_density = volume_field['density'][volume_field['density'] > 0]
        log_limits = np.log10([positive_density.min(), positive_density.max()])
        planar_visualizer = PlotlyVisualizer(
            visualizer_analyzer.sim, reference_system=SOURCE_NAME,
            panel_count=1, perspective='planar', lim=PLOT_LIM, figsize=3,
            shadow_polygon=False, planetstar_connection=True,
            lvl_min=log_limits[0], lvl_max=log_limits[1])
        planar_visualizer.add_grain_field(volume_field)
        planar_visualizer.fig.update_traces(marker_size=5, selector=dict(mode='markers'))
        planar_visualizer.fig.update_yaxes(autorange=False)
        planar_visualizer.set_title('Grain mass density — planar view', color='white')
        #planar_visualizer.fig.write_html(run_path / 'visualizer_planar.html', include_plotlyjs=True)
        planar_visualizer(show_bool=True)
    elif volume_field is not None:
        print('No surviving grains in the requested selection.')

### 6b. Visualizer: line-of-sight (LOS) view

`perspective='los'` displays the same 3D field in the visualizer's −y/z convention, looking along the x direction. We explicitly reverse the horizontal range and keep z upright to match the previous Matplotlib view. It requires `d=3`; an xy-only `d=2` field cannot be reused for this view. Colors still show **local mass density in kg/m³**, not a line integral, extinction, or optical depth. Markers are drawn after the body outlines for visibility; this is not an occultation calculation.

In [ ]:
with archive_directory(run_path):
    if volume_field is not None and volume_field['estimator'] is not None:
        los_visualizer = PlotlyVisualizer(
            visualizer_analyzer.sim, reference_system=SOURCE_NAME,
            panel_count=1, perspective='los', lim=PLOT_LIM, figsize=3,
            lvl_min=log_limits[0], lvl_max=log_limits[1])
        los_visualizer.add_grain_field(volume_field)
        los_visualizer.fig.update_traces(marker_size=5, selector=dict(mode='markers'))
        los_visualizer.fig.update_xaxes(range=list(los_visualizer.fig.layout.xaxis.range)[::-1], autorange=False)
        los_visualizer.fig.update_yaxes(autorange=False)
        los_visualizer.set_title('Grain mass density — LOS view', color='white')
        #los_visualizer.fig.write_html(run_path / 'visualizer_los.html', include_plotlyjs=True)
        los_visualizer(show_bool=True)
    else:
        print('No 3D field available; check the planar cell above.')

In [ ]:
from src.visualizing.webapp import launch_notebook
with archive_directory(run_path):
    analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    app = launch_notebook(analyzer, port=8050)

## 7. Change numerical resolution, not dust production

Repeat with half as many tracers per injection. The injected mass must remain unchanged; sampled sizes, number/area totals, loss fractions, and profile shapes can differ. For a broad distribution, rare large grains can dominate mass normalization. Two short runs at one seed are **not** a convergence study.

In [ ]:
lower_resolution = [replace(population, tracers_per_injection=32) for population in populations]
comparison_path = run_grains(lower_resolution)
with archive_directory(comparison_path):
    comparison_analyzer = SerpensAnalyzer(reference_system=SOURCE_NAME)
    comparison_budget = comparison_analyzer.grain_budget(-1)
    comparison_analyzer.load_timestep_data(-1)
    comparison_profile = comparison_analyzer.grain_radial_profile(
        -1, edges, quantity='mass', center=comparison_analyzer.sim.particles[PRIMARY_NAME].xyz)

np.testing.assert_allclose(comparison_budget['injected']['mass'],
                           sum(b['injected']['mass'] for b in budgets.values()), rtol=1e-12)
assert len(comparison_budget['unaccounted_hashes']) == 0
np.testing.assert_allclose(comparison_budget['retained']['mass'] + comparison_budget['removed']['mass'],
                           comparison_budget['injected']['mass'], rtol=1e-12)
fig = go.Figure()
fig.add_trace(go.Scatter(x=comparison_profile['radii'] / STAR_RADIUS_M,
                         y=comparison_profile['column_density'], mode='lines', line_shape='hvh',
                         name='32 tracers / population / injection'))
fig.add_trace(go.Scatter(x=next(iter(profiles.values()))['radii'] / STAR_RADIUS_M,
                         y=sum(profile['column_density'] for profile in profiles.values()),
                         mode='lines', line_shape='hvh', name='64 tracers / population / injection'))
fig.update_layout(xaxis_title=f'Cylindrical distance / {PRIMARY_NAME} radius',
                  yaxis_title='Total column mass density [kg/m²]')
fig.show()
print(f'Comparison archive: {comparison_path}')

## Before interpreting a torus physically

- **Survival:** fixed radius and composition are assumptions. Even a blackbody temperature estimate is not a sublimation lifetime. Supply material optical/thermal data, vapor-pressure and evaporation laws before estimating `a/|da/dt|`; compare with residence and orbital times. These density and efficiency values are not calibrated silicate or ice.
- **Environment:** estimate gas/plasma drag and charging times from actual densities, temperatures, flows, and composition. A prescribed sphere potential is not self-consistent charging. Nonzero `potential_v` requires `CerpensSimulation`, a rebuilt C backend, `lorentz_enabled=True`, and a suitable magnetic-field configuration; do not just change the potential in this neutral notebook.
- **Optical thinness and collisions:** assess extinction with wavelength-dependent `Q_ext` and the geometry; neither `q_pr` nor geometric area alone gives optical depth. Check grain collision times against residence times.
- **Numerics:** vary duration, timestep, tracer count, seeds, domain size, injection/output cadence, and radial bins. Confirm physical budgets as well as profiles; a smooth plot does not establish convergence.

For a larger monodisperse radius/count comparison, see [the command-line study](../testing/grain_parameter_study.py). The omissions below are also recorded in the grain archive. PR drag is included, not omitted.